In [11]:
import pandas as pd
#df=pd.read_csv("cd.csv")
df=pd.read_csv("cd.csv",index_col="datetime",parse_dates=True)
df.columns
df

,price
datetime,
2001-01-01 09:56:00,72.38
2002-02-02 19:56:00,73.26
2003-03-03 08:56:00,74.56
2004-04-04 07:56:00,41.23
2005-05-05 06:56:00,23.23
2006-06-06 05:56:00,85.16


In [12]:
df.index

DatetimeIndex(['2001-01-01 09:56:00', '2002-02-02 19:56:00',
               '2003-03-03 08:56:00', '2004-04-04 07:56:00',
               '2005-05-05 06:56:00', '2006-06-06 05:56:00'],
              dtype='datetime64[ns]', name='datetime', freq=None)

In [13]:
df.index = pd.to_datetime(df.index)
type(df.index)


pandas.core.indexes.datetimes.DatetimeIndex

In [14]:
df=df.tz_localize(tz='US/Eastern')
df.index

DatetimeIndex(['2001-01-01 09:56:00-05:00', '2002-02-02 19:56:00-05:00',
               '2003-03-03 08:56:00-05:00', '2004-04-04 07:56:00-04:00',
               '2005-05-05 06:56:00-04:00', '2006-06-06 05:56:00-04:00'],
              dtype='datetime64[ns, US/Eastern]', name='datetime', freq=None)

In [15]:
df=df.tz_convert(tz='Europe/Berlin')
df

,price
datetime,
2001-01-01 15:56:00+01:00,72.38
2002-02-03 01:56:00+01:00,73.26
2003-03-03 14:56:00+01:00,74.56
2004-04-04 13:56:00+02:00,41.23
2005-05-05 12:56:00+02:00,23.23
2006-06-06 11:56:00+02:00,85.16


In [16]:
from pytz import all_timezones
all_timezones

['Africa/Abidjan',
 'Africa/Accra',
 'Africa/Addis_Ababa',
 'Africa/Algiers',
 'Africa/Asmara',
 'Africa/Asmera',
 'Africa/Bamako',
 'Africa/Bangui',
 'Africa/Banjul',
 'Africa/Bissau',
 'Africa/Blantyre',
 'Africa/Brazzaville',
 'Africa/Bujumbura',
 'Africa/Cairo',
 'Africa/Casablanca',
 'Africa/Ceuta',
 'Africa/Conakry',
 'Africa/Dakar',
 'Africa/Dar_es_Salaam',
 'Africa/Djibouti',
 'Africa/Douala',
 'Africa/El_Aaiun',
 'Africa/Freetown',
 'Africa/Gaborone',
 'Africa/Harare',
 'Africa/Johannesburg',
 'Africa/Juba',
 'Africa/Kampala',
 'Africa/Khartoum',
 'Africa/Kigali',
 'Africa/Kinshasa',
 'Africa/Lagos',
 'Africa/Libreville',
 'Africa/Lome',
 'Africa/Luanda',
 'Africa/Lubumbashi',
 'Africa/Lusaka',
 'Africa/Malabo',
 'Africa/Maputo',
 'Africa/Maseru',
 'Africa/Mbabane',
 'Africa/Mogadishu',
 'Africa/Monrovia',
 'Africa/Nairobi',
 'Africa/Ndjamena',
 'Africa/Niamey',
 'Africa/Nouakchott',
 'Africa/Ouagadougou',
 'Africa/Porto-Novo',
 'Africa/Sao_Tome',
 'Africa/Timbuktu',
 'Africa/

In [17]:
df=df.tz_convert(tz='Asia/Kolkata')
df

,price
datetime,
2001-01-01 20:26:00+05:30,72.38
2002-02-03 06:26:00+05:30,73.26
2003-03-03 19:26:00+05:30,74.56
2004-04-04 17:26:00+05:30,41.23
2005-05-05 16:26:00+05:30,23.23
2006-06-06 15:26:00+05:30,85.16


In [18]:
df=df.index.tz_convert(tz='Asia/Kolkata')
df

DatetimeIndex(['2001-01-01 20:26:00+05:30', '2002-02-03 06:26:00+05:30',
               '2003-03-03 19:26:00+05:30', '2004-04-04 17:26:00+05:30',
               '2005-05-05 16:26:00+05:30', '2006-06-06 15:26:00+05:30'],
              dtype='datetime64[ns, Asia/Kolkata]', name='datetime', freq=None)

In [19]:
df.index=df.index.tz_convert(tz='Asia/Kolkata')
df

AttributeError: 'DatetimeIndex' object has no attribute 'index'

In [ ]:
⚠️ The error
AttributeError: 'DatetimeIndex' object has no attribute 'index'


means that df.index is already a DatetimeIndex, so you don’t need to use .index on it again.

The mistake was here:

df.index = df.index.tz_convert(tz='Asia/Kolkata')


But this only works if your datetimes already have a timezone.
If not, pandas raises an error saying basically,

"You can’t convert timezone on a naïve datetime (one without tz info)."

✅ Fix depends on your current index state:
👉 Case 1: You already did tz_localize('US/Eastern')

Then conversion is fine — your syntax was correct!

Just run:

df = df.tz_convert('Asia/Kolkata')


✅ You do not assign to df.index, because .tz_convert() returns a new DataFrame with the converted index already.

👉 Case 2: You did not localize yet (index is timezone-naive)

Then you need two steps:

# Step 1: Localize (assign a timezone)
df = df.tz_localize('US/Eastern')

# Step 2: Convert to Asia/Kolkata
df = df.tz_convert('Asia/Kolkata')


Now check:

df.index


Output will look like:

DatetimeIndex(['2001-01-01 20:26:30+05:30', '2001-01-01 20:27:30+05:30'],
              dtype='datetime64[ns, Asia/Kolkata]', name='datetime', freq=None)

✅ Summary
Step	Function	Meaning
tz_localize()	Assign a timezone to naive datetimes	"These times are actually in US/Eastern"
tz_convert()	Convert aware datetimes to another timezone	"Now show them in Asia/Kolkata"

In [20]:
rng=pd.date_range(start='1/1/2017',periods=10,freq='h')
rng

DatetimeIndex(['2017-01-01 00:00:00', '2017-01-01 01:00:00',
               '2017-01-01 02:00:00', '2017-01-01 03:00:00',
               '2017-01-01 04:00:00', '2017-01-01 05:00:00',
               '2017-01-01 06:00:00', '2017-01-01 07:00:00',
               '2017-01-01 08:00:00', '2017-01-01 09:00:00'],
              dtype='datetime64[ns]', freq='h')

In [21]:
rng=pd.date_range(start='1/1/2017',periods=10,freq='h',tz='Europe/London')
rng

DatetimeIndex(['2017-01-01 00:00:00+00:00', '2017-01-01 01:00:00+00:00',
               '2017-01-01 02:00:00+00:00', '2017-01-01 03:00:00+00:00',
               '2017-01-01 04:00:00+00:00', '2017-01-01 05:00:00+00:00',
               '2017-01-01 06:00:00+00:00', '2017-01-01 07:00:00+00:00',
               '2017-01-01 08:00:00+00:00', '2017-01-01 09:00:00+00:00'],
              dtype='datetime64[ns, Europe/London]', freq='h')

In [22]:
rng=pd.date_range(start='1/1/2017',periods=10,freq='h',tz='dateutil/Europe/London')
rng

DatetimeIndex(['2017-01-01 00:00:00+00:00', '2017-01-01 01:00:00+00:00',
               '2017-01-01 02:00:00+00:00', '2017-01-01 03:00:00+00:00',
               '2017-01-01 04:00:00+00:00', '2017-01-01 05:00:00+00:00',
               '2017-01-01 06:00:00+00:00', '2017-01-01 07:00:00+00:00',
               '2017-01-01 08:00:00+00:00', '2017-01-01 09:00:00+00:00'],
              dtype='datetime64[ns, tzfile('GB-Eire')]', freq='h')

In [23]:
rng=pd.date_range(start='1/1/2017 09:20:12',periods=10,freq='30min')
s=pd.Series(range(10),index=rng)
s

2017-01-01 09:20:12    0
2017-01-01 09:50:12    1
2017-01-01 10:20:12    2
2017-01-01 10:50:12    3
2017-01-01 11:20:12    4
2017-01-01 11:50:12    5
2017-01-01 12:20:12    6
2017-01-01 12:50:12    7
2017-01-01 13:20:12    8
2017-01-01 13:50:12    9
Freq: 30min, dtype: int64

In [24]:
b=s.tz_localize(tz='Europe/Berlin')
b

2017-01-01 09:20:12+01:00    0
2017-01-01 09:50:12+01:00    1
2017-01-01 10:20:12+01:00    2
2017-01-01 10:50:12+01:00    3
2017-01-01 11:20:12+01:00    4
2017-01-01 11:50:12+01:00    5
2017-01-01 12:20:12+01:00    6
2017-01-01 12:50:12+01:00    7
2017-01-01 13:20:12+01:00    8
2017-01-01 13:50:12+01:00    9
dtype: int64

In [25]:
b.index

DatetimeIndex(['2017-01-01 09:20:12+01:00', '2017-01-01 09:50:12+01:00',
               '2017-01-01 10:20:12+01:00', '2017-01-01 10:50:12+01:00',
               '2017-01-01 11:20:12+01:00', '2017-01-01 11:50:12+01:00',
               '2017-01-01 12:20:12+01:00', '2017-01-01 12:50:12+01:00',
               '2017-01-01 13:20:12+01:00', '2017-01-01 13:50:12+01:00'],
              dtype='datetime64[ns, Europe/Berlin]', freq=None)

In [26]:
m=s.tz_localize(tz='Asia/Calcutta')
m

2017-01-01 09:20:12+05:30    0
2017-01-01 09:50:12+05:30    1
2017-01-01 10:20:12+05:30    2
2017-01-01 10:50:12+05:30    3
2017-01-01 11:20:12+05:30    4
2017-01-01 11:50:12+05:30    5
2017-01-01 12:20:12+05:30    6
2017-01-01 12:50:12+05:30    7
2017-01-01 13:20:12+05:30    8
2017-01-01 13:50:12+05:30    9
dtype: int64

In [27]:
b+m

2017-01-01 03:50:12+00:00    NaN
2017-01-01 04:20:12+00:00    NaN
2017-01-01 04:50:12+00:00    NaN
2017-01-01 05:20:12+00:00    NaN
2017-01-01 05:50:12+00:00    NaN
2017-01-01 06:20:12+00:00    NaN
2017-01-01 06:50:12+00:00    NaN
2017-01-01 07:20:12+00:00    NaN
2017-01-01 07:50:12+00:00    NaN
2017-01-01 08:20:12+00:00    9.0
2017-01-01 08:50:12+00:00    NaN
2017-01-01 09:20:12+00:00    NaN
2017-01-01 09:50:12+00:00    NaN
2017-01-01 10:20:12+00:00    NaN
2017-01-01 10:50:12+00:00    NaN
2017-01-01 11:20:12+00:00    NaN
2017-01-01 11:50:12+00:00    NaN
2017-01-01 12:20:12+00:00    NaN
2017-01-01 12:50:12+00:00    NaN
dtype: float64